# Sales Forecasting with SSA and Prophet

This notebook demonstrates how to process historical sales data, reduce noise using Singular Spectrum Analysis (SSA), and forecast future sales using Facebook Prophet.

## Steps:
1. **Data Loading & Cleaning**: Convert Excel/CSV and aggregate daily totals.
2. **Singular Spectrum Analysis (SSA)**: Denoise the signal.
3. **Prophet Forecasting**: Predict the next 30 days.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from prophet import Prophet
import warnings
warnings.filterwarnings("ignore")
plt.style.use("ggplot")

### Step 1: Data Preparation
We aggregate the total sales per day and handle any missing dates.

In [ ]:
# Load data
df = pd.read_csv("Sales_Data.csv")

# Clean "TOTAL" rows and convert dates
df = df[~df["DATE"].astype(str).str.contains("TOTAL")]
df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce")
df = df.dropna(subset=["DATE"])

# Aggregate daily totals
df_daily = df.groupby("DATE")["Total"].sum().reset_index()
df_daily = df_daily.sort_values("DATE")

# Re-index to include missing dates with 0 sales
df_daily = df_daily.set_index("DATE").resample("D").sum().reset_index()

print(f"Data points: {len(df_daily)}")
df_daily.head()

### Step 2: Singular Spectrum Analysis (SSA) Implementation
SSA is used to decompose the time series and filter out high-frequency noise while preserving trends.

In [ ]:
class SSA:
    def __init__(self, time_series, L):
        self.series = np.array(time_series)
        self.N = len(self.series)
        self.L = L
        self.K = self.N - L + 1
        
    def embed(self):
        """Embedding Step: Create the Hankel Matrix"""
        X = np.zeros((self.L, self.K))
        for i in range(self.K):
            X[:, i] = self.series[i : i + self.L]
        return X
        
    def decompose(self):
        """Decomposition Step: SVD"""
        X = self.embed()
        U, S, V = np.linalg.svd(X, full_matrices=False)
        return U, S, V
        
    def reconstruct(self, components):
        """Reconstruction Step: Diagonal Averaging"""
        U, S, V = self.decompose()
        X_reconstructed = np.zeros((self.L, self.K))
        for i in components:
            X_reconstructed += S[i] * np.outer(U[:, i], V[i, :])
            
        # Diagonal averaging
        y = np.zeros(self.N)
        L_star, K_star = min(self.L, self.K), max(self.L, self.K)
        for i in range(self.N):
            if 0 <= i < L_star - 1:
                for j in range(i + 1):
                    y[i] += X_reconstructed[j, i - j]
                y[i] /= (i + 1)
            elif L_star - 1 <= i < K_star:
                for j in range(L_star):
                    y[i] += X_reconstructed[j, i - j]
                y[i] /= L_star
            else:
                for j in range(i - K_star + 1, L_star):
                    y[i] += X_reconstructed[j, i - j]
                y[i] /= (self.N - i)
        return y

# Apply SSA
# Window length L (e.g., 14 days for bi-weekly patterns)
L = 14 
ssa_model = SSA(df_daily["Total"], L=L)

# We usually take the first few components (Trend + Major Seasonality)
# To denoise, we exclude high-frequency components (later indices)
df_daily["Denoised_Total"] = ssa_model.reconstruct(components=[0, 1, 2, 3])

plt.figure(figsize=(14, 6))
plt.plot(df_daily["DATE"], df_daily["Total"], label="Original Sales", alpha=0.5)
plt.plot(df_daily["DATE"], df_daily["Denoised_Total"], label="Denoised Sales (SSA)", linewidth=2)
plt.title("Sales Denoising using SSA")
plt.legend()
plt.show()

### Step 3: Prophet Forecasting
We use the denoised signal to train the Prophet model for more stable predictions.

In [ ]:
# Prepare data for Prophet
prophet_df = df_daily[["DATE", "Denoised_Total"]].rename(columns={"DATE": "ds", "Denoised_Total": "y"})

# Initialize and fit
m = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
m.fit(prophet_df)

# Make future forecast (30 days)
future = m.make_future_dataframe(periods=30)
forecast = m.predict(future)

# Plot results
fig1 = m.plot(forecast)
plt.title("30-Day Sales Forecast (Denoised Data)")
plt.show()

# Plot components
fig2 = m.plot_components(forecast)
plt.show()

### Step 4: Final Comparison
Comparing the forecast against the historical noise.

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(df_daily["DATE"], df_daily["Total"], color="gray", alpha=0.3, label="Historical Noise")
plt.plot(forecast["ds"], forecast["yhat"], label="Forecast", color="blue")
plt.fill_between(forecast["ds"], forecast["yhat_lower"], forecast["yhat_upper"], color="blue", alpha=0.1)
plt.title("Final Forecast vs Historical Data")
plt.legend()
plt.show()